In [0]:
# ============================================================
# Dynamic Widgets for Job Cost Monitoring
# Filter by multiple Job IDs, Job Names (or wildcards), or lookback window
# ============================================================

dbutils.widgets.text("job_ids", "")
dbutils.widgets.text("job_names", "clairai_machine_health_compressor, clarai_machine_health_fan, clarai_machine_health_motor, clairai_machine_health_compressor_weekly_run, clarai_machine_health_motor_weekly_run")
dbutils.widgets.text("lookback_days", "30", "3. Lookback Days")


In [0]:
%sql

-- 1. Fact Table: Detailed run-by-run compute & SKU usage
CREATE TABLE IF NOT EXISTS machine_health.prod.pipeline_cost_monitoring (
  job_id             STRING        COMMENT 'Databricks Job ID',
  job_name           STRING        COMMENT 'Pipeline Job Name',
  run_id             STRING        COMMENT 'Databricks Job Run ID',
  sku_name           STRING        COMMENT 'Compute SKU consumed',
  usage_quantity     DECIMAL(38,18) COMMENT 'Total DBUs / compute units consumed',
  usage_unit         STRING        COMMENT 'Usage measurement unit (DBU)',
  estimated_cost_usd DECIMAL(38,6)  COMMENT 'Estimated cost in USD (quantity * list_price)',
  first_usage        TIMESTAMP     COMMENT 'Timestamp of earliest usage within run',
  last_usage         TIMESTAMP     COMMENT 'Timestamp of latest usage within run',
  usage_date         DATE          COMMENT 'Date of usage for partition pruning',
  ingested_at        TIMESTAMP     COMMENT 'Timestamp when record was ingested/updated'
)
USING DELTA
PARTITIONED BY (usage_date)
TBLPROPERTIES (
  'delta.autoOptimize.optimizeWrite' = 'true',
  'delta.autoOptimize.autoCompact'   = 'true'
);

-- 2. Summary Table: Aggregated cost by Job with Total rollup
CREATE TABLE IF NOT EXISTS machine_health.prod.pipeline_cost_summary (
  job_id                   STRING,
  job_name                 STRING,
  total_usage              DECIMAL(38,18),
  estimated_total_cost_usd DECIMAL(38,6),
  run_count                BIGINT,
  earliest_usage           TIMESTAMP,
  latest_usage             TIMESTAMP
);


In [0]:
%sql

-- Incremental Upsert into pipeline_cost_monitoring
MERGE INTO machine_health.prod.pipeline_cost_monitoring AS target
USING (
    SELECT
        u.usage_metadata.job_id AS job_id,
        u.usage_metadata.job_name AS job_name,
        u.usage_metadata.job_run_id AS run_id,
        u.sku_name,
        SUM(u.usage_quantity) AS usage_quantity,
        u.usage_unit,
        SUM(
            u.usage_quantity * p.pricing.`default`
        ) AS estimated_cost_usd,
        MIN(u.usage_start_time) AS first_usage,
        MAX(u.usage_end_time) AS last_usage,
        u.usage_date,
        CURRENT_TIMESTAMP() AS ingested_at
    FROM system.billing.usage u
    JOIN system.billing.list_prices p
        ON u.cloud = p.cloud
        AND u.sku_name = p.sku_name
        AND u.usage_start_time >= p.price_start_time
        AND (
            u.usage_start_time < p.price_end_time
            OR p.price_end_time IS NULL
        )
    WHERE u.billing_origin_product = 'JOBS'
        AND u.usage_date >= DATE_SUB(CURRENT_DATE(), CAST(COALESCE(NULLIF(TRIM(:lookback_days), ''), '30') AS INT))
        -- Filter by Job IDs
        AND (
            COALESCE(TRIM(:job_ids), '') = ''
            OR ARRAY_CONTAINS(
                TRANSFORM(SPLIT(:job_ids, ','), s -> TRIM(s)),
                CAST(u.usage_metadata.job_id AS STRING)
            )
        )
        -- Filter by Job Names or Wildcards
        AND (
            COALESCE(TRIM(:job_names), '') = ''
            OR EXISTS (
                SELECT 1
                FROM EXPLODE(SPLIT(:job_names, ',')) AS t(pattern)
                WHERE LOWER(u.usage_metadata.job_name) LIKE LOWER(TRIM(t.pattern))
            )
        )
    GROUP BY
        u.usage_metadata.job_id,
        u.usage_metadata.job_name,
        u.usage_metadata.job_run_id,
        u.sku_name,
        u.usage_unit,
        u.usage_date
) AS source
ON target.job_id = source.job_id
   AND target.run_id = source.run_id
   AND target.sku_name = source.sku_name
   AND target.usage_date = source.usage_date
WHEN MATCHED AND (
    target.usage_quantity != source.usage_quantity
    OR target.estimated_cost_usd != source.estimated_cost_usd
) THEN UPDATE SET
    target.usage_quantity     = source.usage_quantity,
    target.estimated_cost_usd = source.estimated_cost_usd,
    target.first_usage        = source.first_usage,
    target.last_usage         = source.last_usage,
    target.ingested_at        = source.ingested_at
WHEN NOT MATCHED THEN INSERT (
    job_id,
    job_name,
    run_id,
    sku_name,
    usage_quantity,
    usage_unit,
    estimated_cost_usd,
    first_usage,
    last_usage,
    usage_date,
    ingested_at
) VALUES (
    source.job_id,
    source.job_name,
    source.run_id,
    source.sku_name,
    source.usage_quantity,
    source.usage_unit,
    source.estimated_cost_usd,
    source.first_usage,
    source.last_usage,
    source.usage_date,
    source.ingested_at
);


In [0]:
%sql

-- Populate / Refresh the persistent summary table in machine_health.prod (including ** TOTAL **)
CREATE OR REPLACE TABLE machine_health.prod.pipeline_cost_summary AS
SELECT
    job_id,
    job_name,
    SUM(usage_quantity) AS total_usage,
    SUM(estimated_cost_usd) AS estimated_total_cost_usd,
    COUNT(DISTINCT run_id) AS run_count,
    MIN(first_usage) AS earliest_usage,
    MAX(last_usage) AS latest_usage
FROM machine_health.prod.pipeline_cost_monitoring
WHERE usage_date >= DATE_SUB(CURRENT_DATE(), CAST(COALESCE(NULLIF(TRIM(:lookback_days), ''), '30') AS INT))
    -- Match multiple Job IDs if provided
    AND (
        COALESCE(TRIM(:job_ids), '') = ''
        OR ARRAY_CONTAINS(
            TRANSFORM(SPLIT(:job_ids, ','), s -> TRIM(s)),
            CAST(job_id AS STRING)
        )
    )
    -- Match multiple Job Names or wildcards if provided (case-insensitive)
    AND (
        COALESCE(TRIM(:job_names), '') = ''
        OR EXISTS (
            SELECT 1
            FROM EXPLODE(SPLIT(:job_names, ',')) AS t(pattern)
            WHERE LOWER(job_name) LIKE LOWER(TRIM(t.pattern))
        )
    )
GROUP BY
    job_id,
    job_name

UNION ALL

-- Total row
SELECT
    NULL AS job_id,
    '** TOTAL **' AS job_name,
    SUM(usage_quantity) AS total_usage,
    SUM(estimated_cost_usd) AS estimated_total_cost_usd,
    COUNT(DISTINCT run_id) AS run_count,
    MIN(first_usage) AS earliest_usage,
    MAX(last_usage) AS latest_usage
FROM machine_health.prod.pipeline_cost_monitoring
WHERE usage_date >= DATE_SUB(CURRENT_DATE(), CAST(COALESCE(NULLIF(TRIM(:lookback_days), ''), '30') AS INT))
    AND (
        COALESCE(TRIM(:job_ids), '') = ''
        OR ARRAY_CONTAINS(
            TRANSFORM(SPLIT(:job_ids, ','), s -> TRIM(s)),
            CAST(job_id AS STRING)
        )
    )
    AND (
        COALESCE(TRIM(:job_names), '') = ''
        OR EXISTS (
            SELECT 1
            FROM EXPLODE(SPLIT(:job_names, ',')) AS t(pattern)
            WHERE LOWER(job_name) LIKE LOWER(TRIM(t.pattern))
        )
    );


In [0]:
%sql

-- View detailed run rows from pipeline_cost_monitoring
SELECT
    job_id,
    job_name,
    run_id,
    sku_name,
    usage_quantity,
    usage_unit,
    estimated_cost_usd,
    first_usage,
    last_usage
FROM machine_health.prod.pipeline_cost_monitoring
WHERE usage_date >= DATE_SUB(CURRENT_DATE(), CAST(COALESCE(NULLIF(TRIM(:lookback_days), ''), '30') AS INT))
    AND (
        COALESCE(TRIM(:job_ids), '') = ''
        OR ARRAY_CONTAINS(
            TRANSFORM(SPLIT(:job_ids, ','), s -> TRIM(s)),
            CAST(job_id AS STRING)
        )
    )
    AND (
        COALESCE(TRIM(:job_names), '') = ''
        OR EXISTS (
            SELECT 1
            FROM EXPLODE(SPLIT(:job_names, ',')) AS t(pattern)
            WHERE LOWER(job_name) LIKE LOWER(TRIM(t.pattern))
        )
    )
ORDER BY first_usage DESC;


In [0]:
%sql

-- Query directly from the new pipeline_cost_summary Table
SELECT
    job_id,
    job_name,
    total_usage,
    estimated_total_cost_usd,
    run_count,
    earliest_usage,
    latest_usage
FROM machine_health.prod.pipeline_cost_summary
ORDER BY CASE WHEN job_name = '** TOTAL **' THEN 1 ELSE 0 END, estimated_total_cost_usd DESC;
